# 🌲 Step 1 — Preprocessing & CRS Reprojection
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Generates a synthetic Sentinel-2-style NDVI raster in WGS84 (EPSG:4326), then reprojects it to the
pipeline's CRS contract: **ETRS89 / UTM Zone 29N (EPSG:25829)**. It prints the pixel size before
and after so you can see exactly why this step exists, and exports the reprojected GeoTIFF.

## The Science
Web/lat-long coordinates are **degrees** — a "0.0025° pixel" is 275 m tall but only ~170 m wide at
Irish latitudes, and it shrinks or stretches with latitude. **You cannot compute hectares in
degrees.** UTM 29N is a metric grid: every pixel is a known number of square metres, so
`pixels × pixel_area = hectares` finally works. The pipeline reprojects with `bilinear` resampling
for continuous rasters (NDVI) — categorical maps would use `nearest` instead to avoid inventing
classes.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic NDVI raster in WGS84 (Day-1 ready)
# ============================================================
import numpy as np
import rasterio
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling

np.random.seed(42)
H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)   # SE Ireland test window (lon_min, lat_min, lon_max, lat_max)

def generate_synthetic_ndvi(path, seed=42):
    """Synthetic NDVI-style raster with two forest blobs over the test window."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]
    blob1 = np.exp(-(((xx - 35) / 18) ** 2 + ((yy - 40) / 16) ** 2))
    blob2 = 0.9 * np.exp(-(((xx - 85) / 22) ** 2 + ((yy - 85) / 20) ** 2))
    ndvi = np.clip(0.25 + 0.55 * blob1 + 0.5 * blob2 + rng.normal(0, 0.06, (H, W)), -0.1, 0.95)
    tr = from_bounds(*BOUNDS, W, H)
    with rasterio.open(path, "w", driver="GTiff", height=H, width=W, count=1,
                       dtype="float32", crs="EPSG:4326", transform=tr) as dst:
        dst.write(ndvi.astype("float32"), 1)
    return path

src_tif = outpath("step1_ndvi_wgs84.tif")
generate_synthetic_ndvi(src_tif)
print(f"Synthetic WGS84 raster written: {src_tif}")

In [ ]:
# ============================================================
# Run the reprojection to EPSG:25829 (the pipeline CRS contract)
# ============================================================
import pandas as pd
import matplotlib.pyplot as plt

def reproject_to_utm29n(input_path, output_path):
    """Reproject any raster to EPSG:25829 - mirrors src/crs_handler.py."""
    dst_crs = "EPSG:25829"
    with rasterio.open(input_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds)
        kwargs = src.meta.copy()
        kwargs.update({"crs": dst_crs, "transform": transform, "width": width, "height": height})
        with rasterio.open(output_path, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(source=rasterio.band(src, i), destination=rasterio.band(dst, i),
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=transform, dst_crs=dst_crs,
                          resampling=Resampling.bilinear)
    return output_path

dst_tif = outpath("step1_ndvi_utm29n.tif")
reproject_to_utm29n(src_tif, dst_tif)

with rasterio.open(src_tif) as s, rasterio.open(dst_tif) as d:
    deg_w = s.res[0] * 111320 * np.cos(np.radians(53.0))   # approximate metres at 53°N
    deg_h = s.res[1] * 111320
    report = pd.DataFrame({
        "CRS": ["EPSG:4326 (WGS84 degrees)", "EPSG:25829 (UTM 29N metres)"],
        "Pixel width": [f"~{deg_w:.0f} m", f"{d.res[0]:.1f} m"],
        "Pixel height": [f"~{deg_h:.0f} m", f"{d.res[1]:.1f} m"],
        "Can compute hectares?": ["❌ no - degrees distort", "✅ yes - metric grid"],
    })
print(report.to_string(index=False))

In [ ]:
# ============================================================
# Before / after visual comparison
# ============================================================
with rasterio.open(src_tif) as s, rasterio.open(dst_tif) as d:
    before = s.read(1)
    after = d.read(1)
    ext_b = [s.bounds.left, s.bounds.right, s.bounds.bottom, s.bounds.top]
    ext_a = [d.bounds.left, d.bounds.right, d.bounds.bottom, d.bounds.top]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(before, cmap="RdYlGn", vmin=-0.1, vmax=0.9, extent=ext_b)
axes[0].set_title("Before - EPSG:4326 (degrees)")
axes[1].imshow(after, cmap="RdYlGn", vmin=-0.1, vmax=0.9, extent=ext_a)
axes[1].set_title("After - EPSG:25829 (metres)")
for ax in axes:
    ax.set_xlabel("x"); ax.set_ylabel("y")
plt.tight_layout()

crs_png = outpath("step1_crs_comparison.png")
fig.savefig(crs_png, dpi=120, bbox_inches="tight")
plt.show()

make_download_button(crs_png, label="⬇️ Download comparison chart (.png)", register_as="Step 1 chart")
make_download_button(dst_tif, label="⬇️ Download reprojected GeoTIFF (.tif)", register_as="Step 1 UTM raster")

**Interpretation:** the two panels look almost identical — that is the point. Reprojection is a
coordinate-system change, not a content change. What changed is the **axis units**: the right map
is measured in metres, which is what lets every later phase report area in hectares honestly.
If the right panel had looked warped or empty, that would signal a wrong CRS contract upstream.

In [ ]:
# ============================================================
# Optional: upload your own GeoTIFF and reproject it
# ============================================================
def handle_tif_upload(fname, content_bytes):
    tmp = outpath("uploaded_raster.tif")
    open(tmp, "wb").write(content_bytes)
    try:
        with rasterio.open(tmp) as src:
            print(f"Uploaded '{fname}': {src.width}x{src.height}, {src.count} band(s), CRS = {src.crs}")
    except Exception as e:
        print(f"❌ Could not read '{fname}' as a raster: {e}")
        return
    user_dst = outpath(f"uploaded_utm29n_{fname}")
    try:
        reproject_to_utm29n(tmp, user_dst)
        print(f"✅ Reprojected to EPSG:25829 -> {user_dst}")
        make_download_button(user_dst, label=f"⬇️ Download reprojected '{fname}' (.tif)",
                             register_as=f"Step 1 user raster ({fname})")
    except Exception as e:
        print(f"❌ Reprojection failed: {e}")

make_upload_widget(handle_tif_upload, accept=".tif,.tiff", label="Upload your own .tif")
print("Skip to continue with the synthetic raster.")

# ============================================================
# Regenerate control
# ============================================================
def rerun_step1(seed=None):
    """Fresh synthetic raster + full reprojection, identical logic."""
    global src_tif, dst_tif
    seed = seed or np.random.randint(0, 10_000)
    src_tif = outpath(f"step1_ndvi_wgs84_s{seed}.tif")
    dst_tif = outpath(f"step1_ndvi_utm29n_s{seed}.tif")
    generate_synthetic_ndvi(src_tif, seed=seed)
    reproject_to_utm29n(src_tif, dst_tif)
    print(f"[seed={seed}] regenerated and reprojected -> {dst_tif}")

make_regenerate_button(rerun_step1, label="🔄 Regenerate raster & re-run reprojection")
make_download_all_button()